# 07 — 数据处理：IDF 文件解析

`Tools` 层提供解析 Ivium 专有 **IDF（Ivium 数据格式）**文件的工具。它完全**离线**运行 — 无需 IviumSoft、无需硬件、无需驱动。

### 你可以做什么

- 将 IDF 文件解析为 Python 列表/字典
- 导出数据到 CSV
- 批量转换整个目录的 IDF 文件
- 使用 pandas 和 matplotlib 可视化数据

### IDF 文件结构

一个 IDF 文件可以包含多个数据节：

| 节键 | 内容 |
|-------------|--------|
| `primary_data` | 主测量（始终存在）— 例如 LSV/CV 的 E/I 对，瞬态的时间/I/E，EIS 的 Z1/Z2/freq |
| `ocpdata` | 实验前记录的 OCP 测量 |
| `pretreatmentdata` | 预处理步骤数据 |
| `RsCs_data` | Rs/Cs 测量数据 |
| `osc_data` | 示波器数据（节列表，每个子扫描一节） |

### 测试数据

本笔记本使用 `tests/data/` 中的 IDF 文件 — 它们已提交到仓库，你可以立即运行本笔记本。

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from pyvium import Tools

# 相对于本笔记本定位测试数据
REPO_ROOT = Path("..")
TEST_DATA  = REPO_ROOT / "tests" / "data"

EIS_FILE  = TEST_DATA / "eis_test.idf"
OPEN_FILE = TEST_DATA / "test_open.idf"

print("可用测试文件:")
for f in TEST_DATA.iterdir():
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")

## 1. `get_idf_data()` — 仅主数据

最简函数：仅提取 `primary_data` 节并以行列表形式返回。

In [ ]:
primary = Tools.get_idf_data(str(EIS_FILE))

print(f"类型    : {type(primary)}")
print(f"数据点数: {len(primary)}")
print(f"第[0]行 : {primary[0]}")
print(f"最后行  : {primary[-1]}")

# 每行是浮点数列表 — 列含义取决于方法类型
# EIS：[Re(Z), Im(Z), 频率]
# LSV：[电位, 电流, 0]
# 瞬态：[时间, 电流, 电位]

## 2. `get_all_idf_data()` — 所有节

返回包含文件中所有节的字典。文件中不存在的键直接从字典中省略。

In [ ]:
all_data = Tools.get_all_idf_data(str(EIS_FILE))

print(f"找到的节: {list(all_data.keys())}")
for section, rows in all_data.items():
    if rows:
        print(f"  {section}: {len(rows)} 个点")

## 3. EIS 数据 — Nyquist 图

EIS `primary_data` 行格式为 `[Re(Z), Im(Z), 频率]`。

In [ ]:
eis_df = pd.DataFrame(
    Tools.get_idf_data(str(EIS_FILE)),
    columns=["Z_re", "Z_im", "频率"]
)
print(eis_df.head())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Nyquist 图
axes[0].plot(eis_df["Z_re"], -eis_df["Z_im"], 'o-', markersize=4)
axes[0].set_xlabel("Z' (Ω)")
axes[0].set_ylabel("-Z'' (Ω)")
axes[0].set_title("Nyquist 图")
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# Bode 图 — |Z| 随频率变化
import numpy as np
Z_mag = np.sqrt(eis_df["Z_re"]**2 + eis_df["Z_im"]**2)
phase = np.degrees(np.arctan2(-eis_df["Z_im"], eis_df["Z_re"]))

ax_mag = axes[1]
ax_mag.semilogx(eis_df["频率"], Z_mag, 'b-o', markersize=3, label='|Z|')
ax_mag.set_xlabel("频率 (Hz)")
ax_mag.set_ylabel("|Z| (Ω)", color='b')
ax_mag.set_title("Bode 图")
ax_mag.grid(True, alpha=0.3)

ax_phase = ax_mag.twinx()
ax_phase.semilogx(eis_df["频率"], phase, 'r--o', markersize=3, label='相位')
ax_phase.set_ylabel("相位 (°)", color='r')

plt.tight_layout()
plt.show()

## 4. OCP 数据

如果 IDF 文件包含 OCP 预测量，它出现在 `ocpdata` 键中。行格式通常为 `[时间, 电位, 电流]`。

In [ ]:
all_data = Tools.get_all_idf_data(str(EIS_FILE))

if "ocpdata" in all_data and all_data["ocpdata"]:
    ocp_df = pd.DataFrame(all_data["ocpdata"])
    print(f"OCP 数据: {len(ocp_df)} 个点")
    print(ocp_df.head())

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(ocp_df.iloc[:, 0], ocp_df.iloc[:, 1], 'g-')
    ax.set_xlabel("时间 (s)")
    ax.set_ylabel("OCV (V)")
    ax.set_title("开路电位（预测量）")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("此文件中无 OCP 数据")

## 5. 查看所有节

In [ ]:
for section, rows in all_data.items():
    if not rows:
        print(f"  {section}: 空")
        continue

    # osc_data 是子节列表
    if section == "osc_data":
        print(f"  {section}: {len(rows)} 个子节")
        for i, sub in enumerate(rows):
            print(f"    子节 {i}: {len(sub)} 个点，{len(sub[0]) if sub else 0} 列")
    else:
        cols = len(rows[0]) if rows else 0
        print(f"  {section}: {len(rows)} 个点，{cols} 列")

## 6. 导出到 CSV

`export_to_csv(data, path)` 将任意行列表写入 CSV 文件。

In [ ]:
import tempfile

# 将主数据导出到 CSV
csv_path = os.path.join(tempfile.gettempdir(), "eis_primary.csv")
Tools.export_to_csv(primary, csv_path)
print(f"已导出到: {csv_path}")

# 通过重新读取进行验证
readback = pd.read_csv(csv_path, header=None)
print(f"读回: {len(readback)} 行 × {len(readback.columns)} 列")
print(readback.head())

## 7. `convert_idf_to_csv()` — 一行转换

读取 IDF 文件，提取主数据，并在源文件旁边写入 `.idf.csv` 文件 — 全部在一次调用中完成。

In [ ]:
import shutil

# 在副本上操作，避免修改测试数据
tmp_dir  = Path(tempfile.mkdtemp())
tmp_idf  = tmp_dir / "eis_test.idf"
shutil.copy(EIS_FILE, tmp_idf)

Tools.convert_idf_to_csv(str(tmp_idf))

csv_out = tmp_idf.with_suffix(".idf.csv")
print(f"已创建: {csv_out.name}  ({csv_out.stat().st_size} 字节)")

# 预览
df_check = pd.read_csv(csv_out, header=None)
print(df_check.head())

## 8. `convert_idf_dir_to_csv()` — 批量转换

转换目录中的每个 `.idf` 文件。返回包含计数和所有失败文件名的摘要字典。

In [ ]:
# 将所有测试 IDF 文件复制到临时目录
batch_dir = Path(tempfile.mkdtemp())
for idf in TEST_DATA.glob("*.idf"):
    shutil.copy(idf, batch_dir / idf.name)

print(f"正在转换 {batch_dir} 中的 {len(list(batch_dir.glob('*.idf')))} 个 IDF 文件")

result = Tools.convert_idf_dir_to_csv(str(batch_dir))

print(f"\n结果:")
print(f"  已转换 : {result['converted_count']}")
print(f"  错误   : {result['error_count']}")
if result['errors']:
    print(f"  失败   : {result['errors']}")

print("\n输出文件:")
for f in sorted(batch_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size} 字节)")

## 9. 构建分析流水线

示例：处理目录中的 EIS 文件并从每个文件提取单个量（1 Hz 处的 |Z|）。

In [ ]:
import numpy as np
from pathlib import Path

def extract_z_at_freq(idf_path: str, target_freq: float) -> float:
    """返回最接近 target_freq 频率处的 |Z|。"""
    data = Tools.get_idf_data(idf_path)
    df = pd.DataFrame(data, columns=["Z_re", "Z_im", "频率"])
    idx = (df["频率"] - target_freq).abs().idxmin()
    row = df.iloc[idx]
    return np.sqrt(row["Z_re"]**2 + row["Z_im"]**2)

# 对测试目录中的所有 IDF 文件运行
results = []
for idf_file in sorted(TEST_DATA.glob("*.idf")):
    try:
        z = extract_z_at_freq(str(idf_file), target_freq=1.0)
        results.append({"文件": idf_file.name, "1 Hz 处 |Z| (Ω)": z})
    except Exception as e:
        results.append({"文件": idf_file.name, "1 Hz 处 |Z| (Ω)": None})
        print(f"  跳过 {idf_file.name}: {e}")

summary = pd.DataFrame(results)
print(summary.to_string(index=False))

---

## 小结

| 任务 | 方法 | 说明 |
|------|--------|-------|
| 读取主数据 | `Tools.get_idf_data(path)` | 返回 `List[List[float]]` |
| 读取所有节 | `Tools.get_all_idf_data(path)` | 返回 `Dict[str, List]` |
| 写入 CSV | `Tools.export_to_csv(data, path)` | 任意行列表 |
| 转换单个文件 | `Tools.convert_idf_to_csv(path)` | 在源文件旁创建 `.idf.csv` |
| 批量转换目录 | `Tools.convert_idf_dir_to_csv(dir)` | 返回摘要字典 |

### 数据节列含义

| 节 | 第 0 列 | 第 1 列 | 第 2 列 |
|---------|-------|-------|-------|
| `primary_data`（EIS） | Re(Z) Ω | Im(Z) Ω | 频率 Hz |
| `primary_data`（LSV/CV） | 电位 V | 电流 A | 0 |
| `primary_data`（瞬态） | 时间 s | 电流 A | 电位 V |
| `ocpdata` | 时间 s | 电位 V | 电流 A |

## 下一步

- **`08_batch_and_synchronization.ipynb`** — 使用状态参数协调多台设备